In [1]:
from collections import OrderedDict
from codinglab.shannon_fano_elias import ShannonFanoEliasBinaryCoder
from codinglab import PandasLogger, ProbabilisticSender
from codinglab import ExperimentRunner, NoiselessChannel, TrackingReceiver
import math

logger = PandasLogger()
probabilities = OrderedDict({"a": 0.75, "b": 0.025, "c": 0.1, "d": 0.05, "e": 0.075})

# Create encoder with binary channel alphabet
encoder = ShannonFanoEliasBinaryCoder(probabilities)

# Probabilistic sender
# We will send one symbol per message
sender = ProbabilisticSender(
    encoder=encoder,
    probabilities=probabilities,
    message_length_range=(1, 1),
    logger=logger,
    seed=42,
)

print("Shannon-Fano-Elias Codes:")
for symbol, code in encoder.codes.items():
    print(f"  {symbol}: {code} (length={len(code)})")

print("Running experiment with transmitting 1000 symbols ...")
runner = ExperimentRunner(
    sender, NoiselessChannel(logger=logger), TrackingReceiver(encoder, logger=logger)
)
result = runner.run(num_messages=1000)
shannon_ellias_df = logger.dataframe
shannon_ellias_df.head()

Shannon-Fano-Elias Codes:
  a: 01 (length=2)
  b: 1100001 (length=7)
  c: 11010 (length=5)
  d: 111001 (length=6)
  e: 11110 (length=5)
Running experiment with transmitting 1000 symbols ...


,timestamp,event,message_id,message_data
0,1.773040e+09,source_generated,0,a
1,1.773040e+09,transmitted,0,BinaryAlphabet.zeroBinaryAlphabet.one
2,1.773040e+09,received,0,BinaryAlphabet.zeroBinaryAlphabet.one
3,1.773040e+09,decoded,0,a
4,1.773040e+09,source_generated,1,a


In [15]:
# Энтропия источника
entropy = encoder.entropy
print(f"\nEntropy (H):              {entropy:.4f} bits/symbol")

# Средняя длина кода
avg_length = encoder.expected_code_length
print(f"Expected Length (L):      {avg_length:.4f} bits/symbol")

# Эффективность кода
efficiency = encoder.coding_efficiency
print(f"Coding Efficiency (η):    {efficiency:.4f}")

# Избыточность
redundancy = avg_length - entropy
print(f"Redundancy (L - H):       {redundancy:.4f} bits/symbol")


Entropy (H):              1.2729 bits/symbol
Expected Length (L):      2.8500 bits/symbol
Coding Efficiency (η):    0.4466
Redundancy (L - H):       1.5771 bits/symbol


In [16]:
from codinglab import PrefixCodeTree, TreeNode, PrefixEncoderDecoder

# typing:
from codinglab import SourceChar, ChannelChar
from typing import Sequence, Optional, Dict, List
from dataclasses import dataclass
from enum import Enum

"""
Shannon encoder-decoder implementation for the coding experiments library.

This module implements Shannon coding, a prefix coding algorithm that
assigns codes based on cumulative probabilities. While not optimal,
it provides a simple implementation that guarantees codes of length
ceil(-log2(p)) for symbols with probability p.
"""


def qrepr(p: float, q: int, prec: int) -> list[str]:
    number = int(p * q**prec)
    result = [""] * prec
    for digit in range(prec):
        result[digit] = str((number // (q**digit)) % q)
    return list(reversed(result))


class ShannonEncoder(PrefixEncoderDecoder[SourceChar, ChannelChar]):
    """
    Shannon encoder-decoder for prefix codes.

    This encoder implements the Shannon coding algorithm, which
    constructs prefix codes based on cumulative probabilities of
    symbols sorted in decreasing order of probability. The code
    length for a symbol with probability p is ceil(-log2(p)).

    Attributes:
        _probabilities: Dictionary mapping source symbols to their probabilities
    """

    def __init__(
        self,
        probabilities: Dict[SourceChar, float],
        channel_alphabet: Sequence[ChannelChar],
    ) -> None:
        """Initialize the Shannon encoder with symbol probabilities.

        Args:
            probabilities: Dictionary mapping source symbols to their
                           probabilities (must sum to 1.0)
            channel_alphabet: Sequence of channel symbols for encoding

        Raises:
            ValueError: If probabilities don't sum to approximately 1.0
        """
        # Validate probabilities sum to 1.0
        prob_sum = sum(probabilities.values())
        if not math.isclose(
            prob_sum, 1.0, rel_tol=1e-9
        ):  # Allow for floating-point errors
            raise ValueError(f"Probabilities must sum to 1.0, got {prob_sum}")

        self._probabilities = probabilities
        self._base = len(channel_alphabet)

        super().__init__(probabilities.keys(), channel_alphabet)

    def _build_prefix_code_tree(self) -> None:
        """Build Shannon prefix code tree."""
        # Sort symbols by decreasing probability
        sorted_symbols = sorted(
            self._probabilities.items(),
            key=lambda x: (-x[1], x[0]),  # Sort by probability desc, then symbol
        )

        # Calculate cumulative probabilities
        cumulative = 0.0
        cumulative_probs = []

        for symbol, prob in sorted_symbols:
            cumulative_probs.append((symbol, prob, cumulative))
            cumulative += prob

        # Build prefix code tree
        self._tree = PrefixCodeTree()

        for symbol, prob, cum_prob in cumulative_probs:
            # Calculate code length: ceil(-log2(p))
            if prob > 0:
                code_length = math.ceil(-math.log2(prob))
            else:
                code_length = 0

            # Convert cumulative probability to binary fraction
            # and take first code_length bits
            code = qrepr(cum_prob, self._base, code_length)
            self._tree.insert_code(code, symbol)

        # Build code table from tree
        self._build_table_from_tree()

    @property
    def expected_code_length(self) -> float:
        """
        Calculate the expected code length.

        Returns:
            Expected number of channel symbols per source symbol,
            weighted by symbol probabilities
        """
        if not self._code_table:
            return 0.0

        total = 0.0
        for symbol, prob in self._probabilities.items():
            if symbol in self._code_table:
                total += prob * len(self._code_table[symbol])
        return total

    @property
    def entropy(self) -> float:
        """
        Calculate the Shannon entropy of the source.

        Returns:
            Shannon entropy in bits (for binary channel)
        """
        h = 0.0
        for prob in self._probabilities.values():
            if prob > 0:
                h -= prob * math.log2(prob)
        return h

    @property
    def coding_efficiency(self) -> float:
        """
        Calculate the coding efficiency.

        Returns:
            Ratio of entropy to expected code length,
            representing how close the code is to optimal
        """
        expected_len = self.expected_code_length
        if expected_len == 0:
            return 0.0
        return self.entropy / expected_len

In [17]:
# Сравните избыточность получившегося кода по сравнению с кодом Шенонна и кодом Хаффмана.
# :D
encoder = ShannonEncoder(
    probabilities,
    channel_alphabet=["0", "1"],
)
# Энтропия источника
entropy = encoder.entropy
print(f"\nEntropy (H):              {entropy:.4f} bits/symbol")

# Средняя длина кода
avg_length = encoder.expected_code_length
print(f"Expected Length (L):      {avg_length:.4f} bits/symbol")

# Эффективность кода
efficiency = encoder.coding_efficiency
print(f"Coding Efficiency (η):    {efficiency:.4f}")

# Избыточность
redundancy = avg_length - entropy
print(f"Redundancy (L - H):       {redundancy:.4f} bits/symbol")


Entropy (H):              1.2729 bits/symbol
Expected Length (L):      1.8500 bits/symbol
Coding Efficiency (η):    0.6880
Redundancy (L - H):       0.5771 bits/symbol


In [18]:
import heapq


class BinaryAlphabet(str, Enum):
    zero: "0"
    one: "1"


@dataclass(kw_only=True)
class HuffmanNode(TreeNode[ChannelChar, SourceChar]):
    """Node in the Huffman tree during construction."""

    freq: float

    def __lt__(self, other: "HuffmanNode[ChannelChar, SourceChar]") -> bool:
        return self.freq < other.freq


class BinaryHuffmanEncoder(PrefixEncoderDecoder[SourceChar, BinaryAlphabet]):
    """
    Huffman encoder-decoder for optimal prefix codes.

    This encoder implements the Huffman coding algorithm, which constructs
    an optimal prefix code for a given set of symbol frequencies. More
    frequent symbols get shorter codes, minimizing the expected code length.

    Attributes:
        _frequencies: Dictionary mapping source symbols to their frequencies
    """

    def __init__(self, frequencies: Dict[SourceChar, float]) -> None:
        """Initialize the Huffman encoder with symbol frequencies.

        Args:
            frequencies: Dictionary mapping source symbols to their
                        frequencies (or probabilities)

        Raises:
            ValueError: If frequencies don't match source alphabet,
                       or if channel alphabet is not binary
        """
        self._frequencies = frequencies
        """Dictionary mapping source symbols to their frequencies."""

        super().__init__(frequencies.keys(), ["0", "1"])

    def _build_prefix_code_tree(self) -> None:
        """Build Huffman tree using the priority queue algorithm."""
        # Create leaf nodes for all symbols
        heap = []
        for symbol, freq in self._frequencies.items():
            heapq.heappush(heap, HuffmanNode(freq=freq, value=symbol))

        # Build Huffman tree
        while len(heap) > 1:
            left = heapq.heappop(heap)
            right = heapq.heappop(heap)
            parent = HuffmanNode(
                freq=left.freq + right.freq,
                value=None,
                children={"0": left, "1": right},
            )
            heapq.heappush(heap, parent)

        # Convert Huffman tree to prefix code tree
        root_huffman = heap[0] if heap else None
        self._tree = PrefixCodeTree(root_huffman)
        self._build_table_from_tree()

    def _huffman_to_prefix_tree(
        self, huffman_node: Optional[HuffmanNode]
    ) -> PrefixCodeTree[BinaryAlphabet, SourceChar]:
        """Convert Huffman tree to prefix code tree."""
        prefix_tree = PrefixCodeTree()

        def build_tree(
            current_huffman: HuffmanNode, current_prefix: List[BinaryAlphabet]
        ) -> None:
            if current_huffman.symbol is not None:
                # Leaf node: insert code
                prefix_tree.insert_code(current_prefix, current_huffman.symbol)
            else:
                # Internal node: traverse left (0) and right (1)
                if current_huffman.left:
                    build_tree(
                        current_huffman.left,
                        current_prefix + [self._channel_alphabet[0]],
                    )
                if current_huffman.right:
                    build_tree(
                        current_huffman.right,
                        current_prefix + [self._channel_alphabet[1]],
                    )

        if huffman_node:
            build_tree(huffman_node, [])

        return prefix_tree

    @property
    def expected_code_length(self) -> float:
        """
        Calculate the expected code length.

        Returns:
            Expected number of channel symbols per source symbol,
            weighted by symbol frequencies
        """
        if not self._code_table:
            return 0.0

        total = 0.0
        for symbol, freq in self._frequencies.items():
            if symbol in self._code_table:
                total += freq * len(self._code_table[symbol])
        return total

    @property
    def entropy(self) -> float:
        """
        Calculate the Shannon entropy of the source.

        Returns:
            Shannon entropy in bits (for binary channel)
        """
        h = 0.0
        for freq in self._frequencies.values():
            if freq > 0:
                h -= freq * math.log2(freq)
        return h

    @property
    def coding_efficiency(self) -> float:
        """
        Calculate the coding efficiency.

        Returns:
            Ratio of entropy to expected code length,
            representing how close the code is to optimal
        """
        expected_len = self.expected_code_length
        if expected_len == 0:
            return 0.0
        return self.entropy / expected_len

In [19]:
encoder = BinaryHuffmanEncoder(probabilities)

# Энтропия источника
entropy = encoder.entropy
print(f"\nEntropy (H):              {entropy:.4f} bits/symbol")

# Средняя длина кода
avg_length = encoder.expected_code_length
print(f"Expected Length (L):      {avg_length:.4f} bits/symbol")

# Эффективность кода
efficiency = encoder.coding_efficiency
print(f"Coding Efficiency (η):    {efficiency:.4f}")

# Избыточность
redundancy = avg_length - entropy
print(f"Redundancy (L - H):       {redundancy:.4f} bits/symbol")


Entropy (H):              1.2729 bits/symbol
Expected Length (L):      1.4750 bits/symbol
Coding Efficiency (η):    0.8630
Redundancy (L - H):       0.2021 bits/symbol


Получили

Для Шэннона-Фано-Элиаса: Redundancy (L - H):       1.5771 bits/symbol

Для Шэннона: Redundancy (L - H):       0.5771 bits/symbol

Для Хаффмана: Redundancy (L - H):       0.2021 bits/symbol

In [26]:
import random

# Сравните получившеся фактически размеры закодированных текстов.
# :D
symbols = list(probabilities.keys())
weights = list(probabilities.values())
test_message = "".join(random.choices(symbols, weights=weights, k=1000))

# 1. Shannon-Fano-Elias
encoder = ShannonFanoEliasBinaryCoder(probabilities)
encoded = encoder.encode(test_message)
size = len(encoded)
print(f"Shannon-Fano-Elias text size:       {size} symbols")

# 2. Shannon
encoder = encoder = ShannonEncoder(
    probabilities,
    channel_alphabet=["0", "1"],
)
encoded = encoder.encode(test_message)
size = len(encoded)
print(f"Shannon text size:                  {size} symbols")

# 1. Huffman
encoder = BinaryHuffmanEncoder(probabilities)
encoded = encoder.encode(test_message)
size = len(encoded)
print(f"Huffman text size:                  {size} symbols")

Shannon-Fano-Elias text size:       2827 symbols
Shannon text size:                  1827 symbols
Huffman text size:                  1453 symbols
